# Шаг 1. Загрузка и очистка данных

**Проект:** отмены бронирований в отелях — прогноз и политика предоплаты.

**Цель шага:** получить очищенный датасет, в котором каждое решение по очистке посчитано и обосновано.

**Содержание:** 1. Загрузка · 2. Первичная проверка · 3. Проблемы и решения · 4. Дубли · 5. Запуск очистки и журнал · 6. Устойчивость к обработке дублей · 7. Признаки · 8. Гарантии · 9. Выводы

In [1]:
import pandas as pd

from src.data import load_raw
from src.features import LEAKAGE_COLUMNS
from src.pipeline import build_dataset, check_guarantees, compare_variants, save_outputs

pd.set_option('display.max_columns', 50)

## 1. Загрузка

Датасет Hotel Booking Demand (Antonio, Almeida, Nunes, *Data in Brief*, 2019): брони двух отелей в Португалии — городского (Лиссабон) и курортного (Алгарви), заезды 07.2015–08.2017. Одна строка — один забронированный номер.

In [2]:
raw = load_raw()
raw.shape

(119390, 32)

## 2. Первичная проверка

Типы, пропуски и число уникальных значений по каждой колонке.

In [3]:
overview = pd.DataFrame({
    'тип': raw.dtypes.astype(str),
    'пропусков': raw.isna().sum(),
    'уникальных': raw.nunique(),
})
overview

,тип,пропусков,уникальных
hotel,str,0,2
is_canceled,int64,0,2
lead_time,int64,0,479
arrival_date_year,int64,0,3
arrival_date_month,str,0,12
arrival_date_week_number,int64,0,53
arrival_date_day_of_month,int64,0,31
stays_in_weekend_nights,int64,0,17
stays_in_week_nights,int64,0,35
adults,int64,0,14


**Роли переменных.** Цель — `is_canceled`. Признаки брони: отель, срок брони (`lead_time`), дата заезда, ночи, гости, питание, страна, сегмент и канал продаж, история гостя, тип номера, депозит, агент, компания, лист ожидания, тип клиента, цена за ночь (`adr`), парковка, спецзапросы. **Утечки** — колонки, значения которых известны только после брони или в день заезда; в модель они не попадут:

In [4]:
list(LEAKAGE_COLUMNS)

['reservation_status',
 'reservation_status_date',
 'assigned_room_type',
 'booking_changes']

In [5]:
int(raw.duplicated().sum())

31994

**Вывод.** Пропуски — в `children`, `country`, `agent`, `company`; есть полностью одинаковые строки. Ниже — каждая проблема отдельно: что нашли, решение, почему.

## 3. Проблемы и решения

### 3.1. Ошибки в цене за ночь

In [6]:
raw['adr'].describe()

count    119390.000000
mean        101.831122
std          50.535790
min          -6.380000
25%          69.290000
50%          94.575000
75%         126.000000
max        5400.000000
Name: adr, dtype: float64

In [7]:
extremes = pd.concat([raw.nsmallest(2, 'adr'), raw.nlargest(3, 'adr')])
extremes[['hotel', 'adr', 'adults', 'stays_in_week_nights', 'market_segment']]

,hotel,adr,adults,stays_in_week_nights,market_segment
14969,Resort Hotel,-6.38,2,6,Groups
0,Resort Hotel,0.00,2,0,Direct
48515,City Hotel,5400.00,2,1,Offline TA/TO
111403,City Hotel,510.00,1,1,Offline TA/TO
15083,Resort Hotel,508.00,2,1,Corporate


**Решение:** удалить бронь с отрицательной ценой (шаг 1) и бронь за 5 400 € (шаг 2). **Почему:** цена не бывает отрицательной; 5 400 € в 10 раз дороже следующей брони (510 €) при 2 взрослых на 1 ночь — ошибки ввода. Цена 0 — реальные бесплатные и служебные брони: оставляем, помечаем флагом `is_free` и не используем в анализе цен.

### 3.2. Нереальное число гостей

In [8]:
raw.loc[(raw['adults'] > 10) | (raw['children'] >= 9) | (raw['babies'] >= 9),
        ['adults', 'children', 'babies', 'market_segment', 'is_canceled']]

,adults,children,babies,market_segment,is_canceled
328,2,10.0,0,Offline TA/TO,1
1539,40,0.0,0,Direct,1
1587,26,0.0,0,Offline TA/TO,1
1643,50,0.0,0,Direct,1
1752,26,0.0,0,Offline TA/TO,1
1884,26,0.0,0,Offline TA/TO,1
1917,27,0.0,0,Direct,1
1962,27,0.0,0,Direct,1
2003,26,0.0,0,Offline TA/TO,1
2164,26,0.0,0,Offline TA/TO,1


**Решение:** удалить (шаг 3). **Почему:** в один номер не поселить 10 детей или 55 взрослых — это ошибки ввода или служебные записи.

### 3.3. Брони на 0 ночей и без гостей

In [9]:
nights = raw['stays_in_weekend_nights'] + raw['stays_in_week_nights']
guests = raw['adults'] + raw['children'].fillna(0) + raw['babies']
pd.Series({
    'броней на 0 ночей': int((nights == 0).sum()),
    'из них с ценой 0, %': round(100 * (raw.loc[nights == 0, 'adr'] == 0).mean(), 1),
    'броней без гостей': int((guests == 0).sum()),
})

броней на 0 ночей      715.0
из них с ценой 0, %    100.0
броней без гостей      180.0
dtype: float64

**Решение:** удалить (шаги 4 и 5). **Почему:** бронь без ночей — посещение на день, а не проживание; бронь без гостей не описывает реальное проживание. Объект исследования — номер, в котором собирались ночевать.

### 3.4. Категории Undefined

In [10]:
raw[['meal', 'market_segment', 'distribution_channel']].apply(lambda col: int(col.eq('Undefined').sum()))

meal                    1169
market_segment             2
distribution_channel       5
dtype: int64

**Решение:** питание `Undefined` объединить с `SC` (шаг 7); брони с сегментом или каналом `Undefined` удалить (шаг 6). **Почему:** по документации датасета `Undefined` и `SC` в питании — одно и то же («без пакета питания»); в сегменте и канале это единицы броней без смысла.

### 3.5. Пропуски: страна, агент, компания, дети

In [11]:
raw[['country', 'agent', 'company', 'children']].isna().sum()

country        488
agent        16340
company     112593
children         4
dtype: int64

**Решения:** пустая страна → категория `unknown` (шаг 8): пропуск информативен, страну часто уточняют при заезде. Пустые агент и компания — не пропуск, а «без агента / не от компании»: оставляем, добавляем флаги `has_agent`, `has_company`; агентов с < 100 бронями объединяем в `other` (шаг 9). 4 пропуска в `children` → 0 (шаг 10): 0 — самое частое значение.

### 3.6. Номер недели заезда

In [12]:
months = {m: i for i, m in enumerate(pd.date_range('2016-01-01', periods=12, freq='MS').month_name(), 1)}
arrival = pd.to_datetime(pd.DataFrame({
    'year': raw['arrival_date_year'],
    'month': raw['arrival_date_month'].map(months),
    'day': raw['arrival_date_day_of_month'],
}))
mismatch = arrival.dt.isocalendar().week.to_numpy() != raw['arrival_date_week_number'].to_numpy()
round(100 * mismatch.mean(), 1)

np.float64(53.7)

**Решение:** удалить колонку (шаг 11), неделю при необходимости считать из даты. **Почему:** номер недели расходится с календарной неделей даты заезда — колонка противоречит дате.

## 4. Дубли — главное решение очистки

In [13]:
dup = raw.duplicated()
pd.Series({
    'лишних копий': int(dup.sum()),
    'доля отмен, все строки, %': round(100 * raw['is_canceled'].mean(), 1),
    'доля отмен без дублей, %': round(100 * raw.drop_duplicates()['is_canceled'].mean(), 1),
})

лишних копий                 31994.0
доля отмен, все строки, %       37.0
доля отмен без дублей, %        27.5
dtype: float64

In [14]:
pd.DataFrame({
    'доля копий в сегменте, %': (100 * dup.groupby(raw['market_segment']).mean()).round(1),
    'строк': raw['market_segment'].value_counts(),
}).sort_values('доля копий в сегменте, %', ascending=False)

,"доля копий в сегменте, %",строк
market_segment,,
Groups,75.1,19811
Offline TA/TO,42.7,24219
Corporate,20.5,5295
Online TA,8.6,56477
Direct,6.4,12606
Complementary,5.5,743
Aviation,4.2,237
Undefined,0.0,2


In [15]:
(100 * dup.groupby(raw['deposit_type']).mean()).round(1)

deposit_type
No Deposit    17.6
Non Refund    92.9
Refundable    34.0
dtype: float64

In [16]:
group_sizes = raw.groupby(list(raw.columns), dropna=False).size()
group_sizes[group_sizes > 1].value_counts().sort_index().head(10)

2     5034
3      969
4      497
5      275
6      171
7      115
8      103
9       93
10     123
11      60
Name: count, dtype: int64

**Что нашли.** Копии сосредоточены в групповых продажах (сегмент «Группы», офлайн-агентства) и в депозите «без возврата»; встречаются группы по 3–10+ одинаковых строк. ID брони авторы удалили при обезличивании, поэтому одинаковые строки — скорее **номера одной групповой брони**: одни даты, тариф и условия, а отмена — одно решение группы.

**Варианты:** A — удалить дубли (занижает отмены и выручку, теряем реальные номера); B — оставить (одна групповая отмена считается десятком независимых — тесты завышают значимость); **C — склеить одинаковые строки в одну бронь-решение с числом номеров `n_rooms`**.

**Решение: C.** Тесты и модель работают с независимыми решениями, деньги считаются с весом `n_rooms`, ни один номер не теряется. **Риск:** ошибочно склеить две независимые брони, совпавшие по всем 32 колонкам (вплоть до цены и срока брони до дня) — возможно лишь для части пар в онлайн-продажах. **Страховка:** ниже сравниваем варианты A, B, C; в следующих шагах проверяем на них гипотезы и модель.

## 5. Запуск очистки и журнал

Каждый шаг записывает, сколько строк затронул и почему. Журнал сохраняется в `reports/cleaning_log.csv`.

In [17]:
clean_df, log = build_dataset(raw)
log.to_frame()

,step,action,rule,rows_before,rows_after,rows_affected,reason
0,10,modify,"children: пропуск → 0, тип int",119390,119390,4,Пропусков 4 из 119 тыс.; 0 — самое частое знач...
1,1,remove,Отрицательная цена за ночь,119390,119389,1,Цена не может быть отрицательной — ошибка ввод...
2,2,remove,Цена за ночь ≥ 5000 €,119389,119388,1,Единственная бронь за 5 400 € в 10 раз дороже ...
3,3,remove,"Детей или младенцев > 8, взрослых > 10",119388,119373,15,Столько гостей в один номер не поселить: 10 де...
4,4,remove,Бронь на 0 ночей,119373,118658,715,"715 броней без единой ночи, все с ценой 0 — эт..."
5,5,remove,Бронь без гостей,118658,118548,110,"180 броней без взрослых, детей и младенцев не ..."
6,6,remove,Сегмент или канал продаж Undefined,118548,118543,5,Категория без смысла и слишком мала для отдель...
7,7,modify,Питание Undefined → SC,118543,118543,1160,"По документации датасета (Antonio et al., 2019..."
8,8,modify,Пустая страна → unknown,118543,118543,473,Пропуск информативен — страну часто уточняют п...
9,11,modify,Удалена колонка arrival_date_week_number,118543,118543,118543,Номер недели не совпадает с календарной неделе...


In [18]:
pd.Series({
    'строк в исходных данных': len(raw),
    'удалено строк': log.rows_removed(),
    'номеров после очистки': int(clean_df['n_rooms'].sum()),
    'броней-решений после склейки': len(clean_df),
})

строк в исходных данных         119390
удалено строк                      847
номеров после очистки           118543
броней-решений после склейки     86617
dtype: int64

## 6. Устойчивость к обработке дублей

Доля отмен по броням и по номерам и доля отменённой выручки для трёх вариантов.

In [19]:
compare_variants(raw).round(3)

,bookings,rooms,cancel_rate_bookings,cancel_rate_rooms,canceled_value_share
variant,,,,,
A,86617,86617,0.277,0.277,0.333
B,118543,118543,0.373,0.373,0.392
C,86617,118543,0.277,0.373,0.392


**Как читать.** По номерам вариант C совпадает с B — склейка ничего не теряет. Вариант A занижает долю отмен и отменённой выручки. По броням-решениям доля отмен в C ниже, чем в B: групповые брони больше не умножаются на число номеров.

## 7. Признаки

| Признак | Зачем |
|---|---|
| `arrival_date`, `booking_date` | дата заезда и дата бронирования (заезд − `lead_time`) |
| `nights_week`, `nights_weekend`, `nights_total`, `weekend_share` | длительность и тип поездки |
| `guests`, `is_family` | размер компании, семья с детьми |
| `has_agent`, `has_company`, `agent_id` | бронь через агента / от компании; исходный ID агента |
| `is_free` | бесплатная бронь (цена 0) |
| `n_rooms`, `booking_value` | число номеров и стоимость брони = цена × ночи × номера |
| `arrival_month`, `arrival_weekday`, `season` | сезонность |

In [20]:
clean_df[['booking_key', 'hotel', 'arrival_date', 'booking_date', 'nights_total', 'weekend_share',
          'guests', 'is_family', 'n_rooms', 'booking_value', 'season', 'is_canceled']].head()

,booking_key,hotel,arrival_date,booking_date,nights_total,weekend_share,guests,is_family,n_rooms,booking_value,season,is_canceled
0,0,Resort Hotel,2015-07-01,2015-06-24,1,0.0,1,0,1,75.0,summer,0
1,1,Resort Hotel,2015-07-01,2015-06-18,1,0.0,1,0,1,75.0,summer,0
2,2,Resort Hotel,2015-07-01,2015-06-17,2,0.0,2,0,2,392.0,summer,0
3,3,Resort Hotel,2015-07-01,2015-07-01,2,0.0,2,0,1,214.0,summer,0
4,4,Resort Hotel,2015-07-01,2015-06-22,2,0.0,2,0,1,206.0,summer,0


## 8. Гарантии очищенного датасета

In [21]:
guarantees = pd.Series(check_guarantees(clean_df, len(raw), log), name='выполнено')
assert guarantees.all(), guarantees[~guarantees]
guarantees

1. нет одинаковых строк, n_rooms ≥ 1              True
2. сумма n_rooms = строк после удалений           True
3. ночей ≥ 1, гостей ≥ 1, 0 ≤ adr < порога        True
4. нет Undefined в категориях                     True
5. пропуски только в agent, agent_id и company    True
6. журнал непрерывен                              True
Name: выполнено, dtype: bool

In [22]:
save_outputs(clean_df, log)

## 9. Выводы шага

- Из 119 390 строк исходных данных удалено 847 (0,7%): отрицательная цена за ночь (1), цена 5 400 € (1), нереальное число гостей (15), брони на 0 ночей (715), брони без гостей (110), сегмент или канал продаж Undefined (5).
- После удалений и склейки одинаковых строк осталось 118 543 номера и 86 617 броней-решений: 31 926 строк были копиями одной групповой брони и склеены в `n_rooms`.
- В варианте C доля отменённых номеров — 37,3%, как в варианте B (склейка номеров ничего не теряет), а доля отменённых броней-решений — 27,7%, как в варианте A. Долю отменённой стоимости бронирований C и B оценивают одинаково — 39,2%, тогда как вариант A с удалением дублей занижает её до 33,3%, потому что выбрасывает повторяющиеся номера той же групповой брони вместе с их стоимостью.
- Все 6 гарантий очищенного датасета выполнены: нет одинаковых строк и `n_rooms` ≥ 1, сумма `n_rooms` равна числу строк после удалений, ночи и гости ≥ 1, цена в допустимых пределах, нет категорий Undefined, пропуски остались только в `agent`, `agent_id` и `company`, журнал очистки непрерывен.
- В следующий шаг (EDA и проверка гипотез) передаются: очищенный датасет `data/processed/bookings_clean.parquet`, журнал очистки `reports/cleaning_log.csv` и функция `compare_variants` для сравнения вариантов обработки дублей A, B и C.